### kowiki parser

In [1]:
import os
import sys
import shutil
import json
import glob
import subprocess
import re
import string
from collections import Counter

from tqdm import tqdm

In [2]:
import os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CORPUS_DATA = os.path.join(REPO_ROOT, "data", "1.corpus")

SOURCE_DIR_NAME = "origin" 

# 파일명 (확장자 포함 정확하게)
FILE_NAME = "kowiki_medical.xml" 

# 경로 조립 (상대 경로)
SOURCE_PATH = os.path.join(".", SOURCE_DIR_NAME, FILE_NAME)
DEST_PATH = os.path.join(".", FILE_NAME)       
TEMP_OUTPUT_DIR = "Temp"                      
FINAL_OUTPUT_FILE = os.path.join(CORPUS_DATA, "1.wiki/cleaned/kowiki_dataset_cleaned.txt")
output_dir = os.path.dirname(FINAL_OUTPUT_FILE)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

print(f"찾는 경로: {SOURCE_PATH}")
print(f"저장 경로: {output_dir}")

if os.path.exists(SOURCE_PATH):
    size_bytes = os.path.getsize(SOURCE_PATH)
    size_mb = size_bytes / (1024 * 1024)
    size_gb = size_bytes / (1024 * 1024 * 1024)
    
    print(f"파일명: {FILE_NAME}")
    print(f"크기: {size_mb:.2f} MB ({size_gb:.2f} GB)")
    
else:
    print("X 파일이 없습니다.")
    if os.path.exists(SOURCE_PATH):
        print(f"참고: '{SOURCE_PATH}' 폴더 내 파일 목록:", os.listdir(SOURCE_PATH))
    else:
        print(f"참고: '{SOURCE_PATH}' 폴더 자체가 없습니다.")

찾는 경로: ./origin/kowiki_medical.xml
저장 경로: ./cleaned
파일명: kowiki_medical.xml
크기: 54.81 MB (0.05 GB)


In [3]:
def clean_text_smart(text):
    """
    제시된 clean_str 로직을 적용한 전처리 함수
    1. HTML 태그 제거
    2. 특수문자 매핑 (유사 문자로 치환)
    3. 화이트리스트 필터링 (한글, 영문, 숫자, 구두점 외 제거)
    4. 공백 정리
    """
    # 1. HTML 태그 제거
    text = re.sub(r'<[^>]*>', '', text)

    # 2. 특수문자 매핑 및 제거 로직 정의
    PUNCT_MAPPING = {
        "‘": "'", "₹": "e", "´": "'", "°": "", "€": "e", "™": "tm", "√": " sqrt ", "×": "x", "²": "2",
        "—": "-", "–": "-", "’": "'", '”': '"', '“': '"', "£": "e", '∞': 'infinity',
        'θ': 'theta', '÷': '/', 'α': 'alpha', '•': '.', 'à': 'a', '−': '-', 'β': 'beta', '∅': '', '³': '3', 'π': 'pi',
        '\u200b': ' ', '…': ' ... ', '\ufeff': '', 'करना': '', 'है': '', '·': '.'
    }

    # 매핑 적용
    for p, v in PUNCT_MAPPING.items():
        text = text.replace(p, v)

    # 3. 화이트리스트 필터링
    # string.punctuation의 모든 문자를 정규식에서 안전하게 인식되도록 이스케이프 처리
    escaped_punct = re.escape(string.punctuation)
    
    # 허용할 문자: 영문(A-Za-z), 숫자(0-9), 한글(자모 포함), 구두점, 공백(\s)
    # ^(not)을 사용하여 이 목록에 없는 문자는 빈 문자열로 치환
    text = re.sub(f"[^A-Za-z0-9ㄱ-ㅣ가-힣{escaped_punct}\\s]", "", text)

    # 4. 다중 공백을 하나의 공백으로 치환하고 앞뒤 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [4]:
# ==========================================
# 1. 파일 복사
# ==========================================
def copy_with_progress(src, dst):
    if not os.path.exists(src):
        print(f"❌ Error: Source file not found at {src}")
        return False
        
    if os.path.exists(dst):
        print(f"ℹ️ File already exists at {dst}. Skipping copy.")
        return True

    file_size = os.path.getsize(src)
    
    with open(src, 'rb') as fsrc:
        with open(dst, 'wb') as fdst:
            with tqdm(total=file_size, unit='B', unit_scale=True, desc=f"Copying {FILE_NAME}") as pbar:
                while True:
                    chunk = fsrc.read(1024 * 1024)
                    if not chunk:
                        break
                    fdst.write(chunk)
                    pbar.update(len(chunk))
    return True

print(f"\n[Step 1/3] Copying XML dump...")

# SOURCE_PATH, DEST_PATH 등은 이전 셀에서 설정되었다고 가정
if copy_with_progress(SOURCE_PATH, DEST_PATH):
    
    # ==========================================
    # 2. XML 파싱 (WikiExtractor 실행)
    # ==========================================
    print(f"\n[Step 2/3] Extracting text from XML (This takes time)...")
    
    cmd = [
        sys.executable, "-m", "wikiextractor.WikiExtractor",
        DEST_PATH,
        "--json", 
        "-o", TEMP_OUTPUT_DIR,
        "--processes", str(max(1, os.cpu_count() - 1))
    ]
    
    try:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        stdout, stderr = process.communicate()
        
        if process.returncode == 0:
            print("✅ Extraction completed successfully.")
        else:
            print(f"❌ Error during extraction:\n{stderr}")
            raise Exception("WikiExtractor failed")
            
    except Exception as e:
        print(f"Execution failed: {e}")

    # ==========================================
    # 3. JSON 병합 -> TXT 저장
    # ==========================================
    print(f"\n[Step 3/3] Merging extracted files into {FINAL_OUTPUT_FILE}...")
    
    files = glob.glob(f"{TEMP_OUTPUT_DIR}/**/wiki_*", recursive=True)
    
    if not files:
        print("❌ No extracted files found. Check the extraction step.")
    else:
        with open(FINAL_OUTPUT_FILE, 'w', encoding='utf-8') as out_f:
            for file_path in tqdm(files, desc="Merging Files"):
                with open(file_path, 'r', encoding='utf-8') as in_f:
                    for line in in_f:
                        try:
                            # 중간 파일은 JSON 형식이므로 로드 후 text만 추출
                            data = json.loads(line)
                            raw_text = data['text']
                            
                            cleaned_text = clean_text_smart(raw_text)
                            
                            # 50자 미만 문서는 버림
                            if len(cleaned_text) < 50:
                                continue
                            
                            # [핵심 변경] JSON 구조 없이 텍스트만 기록
                            # 문서 간 구분은 개행 문자 두 개(\n\n) 사용
                            out_f.write(cleaned_text + '\n\n')
                            
                        except json.JSONDecodeError:
                            continue
                            
        print(f"\n🎉 All Done! Final dataset saved to: {os.path.abspath(FINAL_OUTPUT_FILE)}")
        
        # 샘플 확인 (TXT 읽기 방식)
        print("\n--- First 200 characters of the first document ---")
        if os.path.exists(FINAL_OUTPUT_FILE):
            with open(FINAL_OUTPUT_FILE, 'r', encoding='utf-8') as f:
                # 처음 200글자만 읽어서 출력
                print(f.read(200))
else:
    print("Process stopped due to copy failure.")


[Step 1/3] Copying XML dump...
ℹ️ File already exists at ./kowiki_medical.xml. Skipping copy.

[Step 2/3] Extracting text from XML (This takes time)...
✅ Extraction completed successfully.

[Step 3/3] Merging extracted files into ./cleaned/kowiki_dataset_cleaned.txt...


Merging Files: 100%|████████████████████████████████████████████████████████████████████| 49/49 [00:01<00:00, 28.09it/s]


🎉 All Done! Final dataset saved to: ./datasets/1.Wiki/cleaned/kowiki_dataset_cleaned.txt

--- First 200 characters of the first document ---
RNA 또는 리보핵산()은 기능 그 자체를 수행하거나(비번역 RNA) 단백질 생성을 위한 정보 분자로써(mRNA) 대부분의 생물학적 기능에 필수적인 고분자 화합물이다. RNA(리보핵산)과 DNA(디옥시리보핵산)는 핵산이다. 핵산은 알려진 모든 형태의 생명체에 필수적인 4가지 주요 거대분자 중 하나를 구성한다. RNA는 뉴클레오타이드의 사슬로 조립된다. 생물


In [5]:
def analyze_noise_characters_txt(file_path, top_n=50):
    """
    TXT 파일을 줄 단위로 읽어 한글, 영어, 공백을 제외한 문자들의 빈도를 계산합니다.
    (JSON 파싱 과정 없이 텍스트 자체를 분석)
    """
    print(f"\n[Analysis] Counting non-Korean/non-English characters in {file_path}...")
    
    # 분석에서 '정상'으로 간주하고 제거할 패턴
    # 남는 것: 숫자, 구두점, 특수문자, 한자, 이모지 등
    allowed_pattern = re.compile(r'[가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z\s]+')
    
    noise_counter = Counter()
    
    if not os.path.exists(file_path):
        print(f"❌ Error: File not found at {file_path}")
        return

    total_lines = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Analyzing Noise"):
            total_lines += 1
            
            # JSON 파싱 없이 바로 정규식 적용
            # 정상 문자열(한글/영어/공백)을 지우고 남은 것(노이즈)만 추출
            noise_chars = allowed_pattern.sub('', line)
            
            if noise_chars:
                noise_counter.update(noise_chars)

    # 결과 출력
    print(f"\n📊 Top {top_n} Other Characters (Potential Noise):")
    print("-" * 55)
    print(f"{'Rank':^5} | {'Char':^10} | {'Count':^12} | {'Hex Code':^10}")
    print("-" * 55)
    
    if not noise_counter:
        print("No noise characters found. The text contains only Korean, English, and Whitespace.")
    
    for rank, (char, count) in enumerate(noise_counter.most_common(top_n), 1):
        # 1. 출력 불가능하거나 공백인 문자는 눈에 보이게 표현
        char_display = repr(char) if not char.isprintable() or char.strip() == '' else char
        # 2. 유니코드 값을 16진수로 표현
        hex_code = f"U+{ord(char):04X}"
        
        print(f"{rank:^5} | {char_display:^10} | {count:^12,} | {hex_code:^10}")

    print("-" * 55)
    print(f"Total Lines Processed: {total_lines:,}")

# 실행
if os.path.exists(FINAL_OUTPUT_FILE):
    analyze_noise_characters_txt(FINAL_OUTPUT_FILE)


[Analysis] Counting non-Korean/non-English characters in ./cleaned/kowiki_dataset_cleaned.txt...


Analyzing Noise: 9988it [00:00, 23801.44it/s]


📊 Top 50 Other Characters (Potential Noise):
-------------------------------------------------------
Rank  |    Char    |    Count     |  Hex Code 
-------------------------------------------------------
  1   |     .      |   198,562    |   U+002E  
  2   |     ,      |   126,694    |   U+002C  
  3   |     )      |    71,121    |   U+0029  
  4   |     (      |    70,994    |   U+0028  
  5   |     1      |    68,439    |   U+0031  
  6   |     2      |    35,623    |   U+0032  
  7   |     0      |    34,934    |   U+0030  
  8   |     9      |    29,978    |   U+0039  
  9   |     5      |    20,405    |   U+0035  
 10   |     3      |    20,185    |   U+0033  
 11   |     4      |    17,835    |   U+0034  
 12   |     8      |    17,609    |   U+0038  
 13   |     6      |    16,510    |   U+0036  
 14   |     "      |    14,459    |   U+0022  
 15   |     7      |    13,985    |   U+0037  
 16   |     '      |    12,514    |   U+0027  
 17   |     -      |    9,880     |   U+002